In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# Data Preprocessing

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("duongtran1909/wiki727k")

print("Path to dataset files:", path)

In [ ]:
import os
os.listdir(path + '/wiki_727')

In [ ]:
import os

# Base directory containing the documents
base_path = path + '/wiki_727/train/'

# List to store all document paths
all_documents = []

# Use os.walk to recursively go through all directories and subdirectories
for root, dirs, files in os.walk(base_path):
    # Iterate over the files in each directory
    for file in files:
        # Create the full file path and add it to the list
        file_path = os.path.join(root, file)
        all_documents.append(file_path)

# Sort the list of documents if you want a consistent order
all_documents.sort()

# Print the first few file paths for verification
for doc in all_documents[:10]:
    print(doc)
print(len(all_documents))


In [ ]:
from tqdm import tqdm  # Import tqdm for progress bar

# Function to read the content of a file
def read_file_from_path(filepath):
    """Reads a file from the specified path and returns its content.

    Args:
        filepath: The path to the file.

    Returns:
        The content of the file as a string, or None if the file does not exist.
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as file:
            content = file.read()
        return content
    except FileNotFoundError:
        print(f"Error: File not found at '{filepath}'")
        return None

# Function to split a file into sentences
def split_file_into_sentences(filepath):
    """Reads a file, splits it into lists of sentences by "========", and returns the lists."""
    file_content = read_file_from_path(filepath)
    if file_content:
        lists_of_sentences = []
        sentences = file_content.split("========")
        for sentence in sentences:
            lists_of_sentences.append(sentence.strip())  # Remove leading/trailing whitespaces
        return lists_of_sentences
    else:
        return None

# Process half of the files
all_sentences = []  # List to store sentences from all documents

# Determine the number of files to process (half of the total)
num_files_to_process = len(all_documents) // 3

# Loop through half of the files with a progress bar
for file_path in tqdm(all_documents[:num_files_to_process], desc="Processing files", unit="file"):
    sentences = split_file_into_sentences(file_path)
    if sentences:
        sent_file = []
        for i, sentence_list in enumerate(sentences):
            if i == 0:
                continue  # Skip the first sentence (or modify as per your logic)
            sent_file.append(sentence_list.split('\n')[1:])  # Assuming each sentence is split by newline

        # Append sentences from this file to the all_sentences list
        all_sentences.extend(sent_file)




In [ ]:
import csv

def convert_to_csv(all_sentences, output_filename="output.csv"):
    """Converts a list of sentences to a CSV file with labels.

    Args:
        all_sentences: A list of lists, where each inner list represents a file
                       and contains sentences.
        output_filename: The name of the output CSV file.
    """
    with open(output_filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['sentence', 'label'])  # Header row

        # Process each sentence list from all the files
        for sentence_list in all_sentences:
            for i, sentence in enumerate(sentence_list):
                label = 1 if i == 0 else 0  # Label the first sentence as 1, others as 0
                writer.writerow([sentence, label])

# Call the function to convert all sentences to a CSV file
convert_to_csv(all_sentences)


In [ ]:
import pandas as pd

csv_file_path = '/content/output.csv'
df = pd.read_csv(csv_file_path)
print(df['sentence'][0])


In [ ]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Define source file path and destination directory
source_file = '/content/output.csv'
destination_dir = '/content/drive/My Drive/'  # Change this path as needed

# Copy the file to Google Drive
shutil.copy(source_file, destination_dir)

print(f"File moved to: {destination_dir}")


## Data Windowing

In [ ]:
import pandas as pd

# Assuming the file exists and is accessiblea
try:
    df = pd.read_csv('/content/drive/MyDrive/output.csv')
    print(df.iloc[0])  # Print the first row (element 0)
except FileNotFoundError:
    print("File not found at the specified path.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
for i,item in enumerate(df['sentence']):
  if type(item) != str:
    print(i)


In [ ]:
# drop none index
df = df.drop(index=1017419)

In [ ]:
import os
import json
from tqdm import tqdm

output_dir = '/content/window_tokenization'
w_number = 0
for i in tqdm(range(0,len(df)//2,8)):
  s = dict()
  s['labels'] = list(df['label'][i:i+16])
  encoded = tokenizer_1(
        list(df['sentence'][i:i+16]), padding='max_length', truncation=True, max_length=128, return_tensors="pt"
    )
  s['input_ids'] = encoded['input_ids'].tolist()
  s['attention_mask']= encoded['attention_mask'].tolist()
  s['sentences_index'] = list(range(i,i+16))

  # Create a filename based on the index
  filename = os.path.join(output_dir, f"windows_{w_number}.json")
  w_number += 1
  # Save the dictionary as a JSON file
  with open(filename, 'w') as f:
      json.dump(s, f, indent=4) # Use indent for pretty printing



In [ ]:
import shutil
import os
from tqdm import tqdm
from google.colab import drive


def zip_directory(directory_path, zip_filename):
    """Zips a directory using shutil and tqdm for progress visualization."""
    try:
        with tqdm(total=len(os.listdir(directory_path)), desc="Zipping directory") as pbar:
          shutil.make_archive(zip_filename, 'zip', directory_path)
          pbar.update(len(os.listdir(directory_path)))  # Update progress bar
        print(f"Directory '{directory_path}' zipped to '{zip_filename}.zip'")

    except FileNotFoundError:
        print(f"Error: Directory '{directory_path}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage: Replace with your desired directory path and zip file name
directory_to_zip = '/content/window_tokenization'  # Path to directory you want to zip
zip_file_name = '/content/zipped_windows' # Path where you want to store the zip file
zip_directory(directory_to_zip, zip_file_name)

In [ ]:
# Copy the zipped file to your Google Drive
!cp /content/zipped_windows.zip /content/drive/MyDrive/

In [ ]:
# prompt: print dir size

import os

def get_directory_size(path):
  """Returns the size of a directory in bytes."""
  total_size = 0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      fp = os.path.join(dirpath, f)
      # skip if it is symbolic link
      if not os.path.islink(fp):
        total_size += os.path.getsize(fp)
  return total_size

# Example usage: Replace with your desired directory path
directory_path = '/content/window_tokenization'
size_in_bytes = get_directory_size(directory_path)

print(f"Size of directory '{directory_path}': {size_in_bytes} bytes")

# Convert to other units if needed (e.g., kilobytes, megabytes, gigabytes)
size_in_kb = size_in_bytes / 1024
size_in_mb = size_in_kb / 1024
size_in_gb = size_in_mb / 1024


print(f"Size of directory '{directory_path}': {size_in_kb:.2f} KB")
print(f"Size of directory '{directory_path}': {size_in_mb:.2f} MB")
print(f"Size of directory '{directory_path}': {size_in_gb:.2f} GB")

# Dataset Class

In [ ]:
import zipfile
import os


with zipfile.ZipFile('/content/drive/MyDrive/zipped_windows.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/data')


In [ ]:
Datasize = len(os.listdir('/content/data'))

In [ ]:
import torch
from torch.utils.data import Dataset
import json
import os

class JsonDataset(Dataset):
    def __init__(self, file_path):
        self.data_path = file_path

    def __len__(self):
        return Datasize-10000

    def __getitem__(self, idx):
        file_path = self.data_path+f"windows_{idx}.json"
        with open(file_path, 'r') as f:
          data = json.load(f)
          input_ids = torch.tensor(data['input_ids'])
          attention_mask = torch.tensor(data['attention_mask'])
          labels = torch.tensor(data['labels'])
        return input_ids, attention_mask, labels


class TestJsonDataset(Dataset):
    def __init__(self, file_path):
        self.data_path = file_path

    def __len__(self):
        return 10000

    def __getitem__(self, idx):
        file_path = self.data_path+f"windows_{Datasize-10001+idx}.json"
        with open(file_path, 'r') as f:
          data = json.load(f)
          input_ids = torch.tensor(data['input_ids'])
          attention_mask = torch.tensor(data['attention_mask'])
          labels = torch.tensor(data['labels'])
        return input_ids, attention_mask, labels


In [ ]:
dataset = JsonDataset('/content/data/')
dataset_test = TestJsonDataset('/content/data/')

#Model

In [ ]:
import torch
import torch.nn as nn

class Adapter(nn.Module):
    def __init__(self, hidden_size, adapter_size=64):
        """
        Initialize the adapter layer.
        Args:
        - hidden_size: The hidden size of the transformer model (H).
        - adapter_size: The reduced size of the adapter bottleneck (a).
        """
        super(Adapter, self).__init__()
        self.down_project = nn.Linear(hidden_size, adapter_size)  # W_d
        self.activation = nn.GELU()  # g(·), can use GELU if preferred
        self.up_project = nn.Linear(adapter_size, hidden_size)  # W_u

    def forward(self, x):
        """
        Forward pass for the adapter.
        Args:
        - x: Input tensor of shape (batch_size, seq_length, hidden_size).
        Returns:
        - Output tensor of the same shape as input (residual connection applied).
        """
        # Down-project, apply activation, and up-project

        down = self.down_project(x)  # Shape: (batch_size, seq_length, adapter_size)
        activated = self.activation(down)  # Shape: (batch_size, seq_length, adapter_size)
        up = self.up_project(activated)  # Shape: (batch_size, seq_length, hidden_size)

        # Add residual connection
        return x + up  # Shape: (batch_size, seq_length, hidden_size)

class RobertaInput(nn.Module):
    def __init__(self, adapter_size=64):
        super(RobertaInput, self).__init__()
        self.roberta_embeddings = RobertaModel.from_pretrained("roberta-base").embeddings

    def forward(self, input_ids, attention_mask=None):
        batch_size,window_size,tokens_ids = input_ids.shape

        input_ids = input_ids.view(-1, tokens_ids)
        if attention_mask is not None:
          attention_mask = attention_mask.view(-1, tokens_ids).unsqueeze(-1)

        token_embeddings = self.roberta_embeddings(input_ids=input_ids)

        embeddings = token_embeddings * attention_mask
        embeddings = embeddings.sum(dim=1)  # Sum over tokens
        sum_mask = attention_mask.sum(dim=1)  # Sum of attention mask (to normalize)

        sentence_embeddings = embeddings / sum_mask
        sentence_embeddings = sentence_embeddings.view(batch_size, window_size, -1)  # Shape: (batch_size, window_size, hidden_size)

        return sentence_embeddings

class RobertaEncoderWithAdapters(nn.Module):
    def __init__(self, adapter_size=64):
        super(RobertaEncoderWithAdapters, self).__init__()
        roberta = RobertaModel.from_pretrained("roberta-base")
        self.ATTEN = roberta.encoder.layer[0].attention
        self.FFN_1 = roberta.encoder.layer[0].intermediate
        self.FFN_2 = roberta.encoder.layer[0].output
        self.ADP = Adapter(hidden_size=roberta.config.hidden_size)

    def forward(self, input, attention_mask=None):
        # Forward through RoBERTa layers with adapters
        if attention_mask is not None:
          attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
          attention_mask = attention_mask.view(-1, 128)
        temp = self.ATTEN(input,attention_mask=attention_mask)
        temp = temp[0]
        temp = self.ADP(temp)
        temp = self.FFN_1(temp)
        temp = self.FFN_2(temp,input) #adding process
        temp = self.ADP(temp)
        return temp


class LowerTokernizer(nn.Module):
    def __init__(self, num_of_layer=12):
        super(LowerTokernizer, self).__init__()
        self.Embd = RobertaInput()
        self.Encoder = nn.ModuleList([RobertaEncoderWithAdapters() for _ in range(num_of_layer)])

    def forward(self, input_ids, attention_mask=None):
        """
        Processes 3D input directly.
        Args:
        - input_ids: Tensor of shape (batch_size, window_size, tokens_ids)
        - attention_mask: Tensor of shape (batch_size, window_size, tokens_ids), optional.
        Returns:
        - output: Tensor of shape (batch_size, window_size, tokens_ids, hidden_size)
        """

        # Embeddings
        temp = self.Embd(input_ids=input_ids, attention_mask=attention_mask)

        # Process each token window independently
        # if attention_mask is not None:
        #     attention_mask = attention_mask.view(batch_size * window_size, tokens_ids)

        # Pass through each transformer layer
        for layer in self.Encoder:
            temp = layer(temp)

        # Reshape back to 4D
        # temp = temp.view(batch_size, window_size, tokens_ids, hidden_size)
        return temp

    def _initialize_roberta_weights(self):
        print("Initializing weights from pre-trained RoBERTa model...")

        roberta = RobertaModel.from_pretrained("roberta-base")
        print("Loaded RoBERTa model.")

        # Initialize embedding weights
        self.Embd.roberta_embeddings.load_state_dict(roberta.embeddings.state_dict())
        print("Embeddings initialized.")

        # Loop through all layers of your custom encoder and copy the weights
        for i, encoder_layer in enumerate(self.Encoder):
            # Load weights for the attention layer
            encoder_layer.ATTEN.load_state_dict(roberta.encoder.layer[i].attention.state_dict())
            print(f"Layer {i}: Attention weights initialized with shape:")
            for name, param in roberta.encoder.layer[i].attention.state_dict().items():
                print(f"  {name}: {param.shape}")

            # Load weights for the feed-forward layers (intermediate and output)
            encoder_layer.FFN_1.load_state_dict(roberta.encoder.layer[i].intermediate.state_dict())
            print(f"Layer {i}: FFN_1 weights initialized with shape:")
            for name, param in roberta.encoder.layer[i].intermediate.state_dict().items():
                print(f"  {name}: {param.shape}")

            encoder_layer.FFN_2.load_state_dict(roberta.encoder.layer[i].output.state_dict())
            print(f"Layer {i}: FFN_2 weights initialized with shape:")
            for name, param in roberta.encoder.layer[i].output.state_dict().items():
                print(f"  {name}: {param.shape}")

        # Freeze the embedding weights
        for param in self.Embd.parameters():
            param.requires_grad = False
        print("Embedding weights frozen.")

        # Freeze the weights in the encoder layers
        for i, encoder_layer in enumerate(self.Encoder):
            for param in encoder_layer.ATTEN.parameters():
                param.requires_grad = False
            for param in encoder_layer.FFN_1.parameters():
                param.requires_grad = False
            for param in encoder_layer.FFN_2.parameters():
                param.requires_grad = False
            print(f"Layer {i}: Weights frozen.")

        print("Weight initialization and freezing completed.")


In [ ]:
import torch
import torch.nn as nn
from transformers import RobertaModel

class UpperTokernizer(nn.Module):
    def __init__(self, num_of_layer=6):  # Add num_classes argument
        super(UpperTokernizer, self).__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.last_six_layers = nn.ModuleList([self.roberta.encoder.layer[i] for i in range(12 - num_of_layer, 12)])
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(self.roberta.config.hidden_size, 16)  # Output layer for num_classes
        self.softmax = nn.Softmax(dim=-1)  # Softmax activation

    def forward(self, input_embeddings, attention_mask=None):
        temp = input_embeddings
        print
        for layer in self.last_six_layers:
            temp = layer(temp, attention_mask=attention_mask)[0]

        pooled_output = self.pooling(temp.permute(0, 2, 1)).squeeze(-1)
        output = self.fc(pooled_output)
        output = self.softmax(output)  # Apply softmax
        return output

In [ ]:
class BigModel(nn.Module):
    def __init__(self):
        super(BigModel, self).__init__()
        self.lower_tokenizer = LowerTokernizer(num_of_layer=12)
        self.lower_tokenizer._initialize_roberta_weights()
        self.upper_tokenizer = UpperTokernizer(num_of_layer=6)

    def forward(self, input_ids, attention_mask=None):
        lower_output = self.lower_tokenizer(input_ids=input_ids, attention_mask=attention_mask)
        upper_output = self.upper_tokenizer(lower_output, attention_mask=None)

        return upper_output


#Training code

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import numpy as np
import os

# Mount Google Drive (Uncomment if using Google Colab)
# from google.colab import drive
# drive.mount('/content/drive')

# Define the path to save the model
save_path = '/content/drive/My Drive/models/'  # Replace with your preferred path
os.makedirs(save_path, exist_ok=True)

# Hyperparameters
learning_rate = 1e-5
batch_size = 32
epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data Loader
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
Test_dataloader = DataLoader(dataset_test, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)

# Model, Loss, and Optimizer
model = BigModel().to(device)
criterion = nn.BCELoss()  # Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Early Stopping
patience = 3
best_val_loss = 0.3462
epochs_no_improve = 0

checkpoint_path = "/content/drive/MyDrive/models/best_model.pth"
model.load_state_dict(torch.load(checkpoint_path))



# Training Loop
for epoch in range(2,4):
    model.train()  # Set the model to training mode
    epoch_loss = 0
    for input_ids, attention_mask, labels in tqdm(dataloader):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device).float()  # Ensure labels are float

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # Validation (example: split your data into train and validation sets)
    # Replace this with your actual validation logic
    model.eval()
    with torch.no_grad():
        val_loss = 0.0
        for input_ids, attention_mask, labels in Test_dataloader:  # Replace with your validation Test_dataloader
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device).float()  # Ensure labels are float
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    val_loss /= len(Test_dataloader)
    print(f'Epoch {epoch+1}/{epochs}, Train Loss: {epoch_loss/len(dataloader):.4f}, Validation Loss: {val_loss:.4f}')
    # Assuming these variables and objects are defined: model, optimizer, criterion, val_loss, best_val_loss, epochs_no_improve, save_path

    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': val_loss,
    }

    # Save the best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(checkpoint, os.path.join(save_path, 'best_model.pth'))
        print(f"Best model saved at epoch {epoch+1} with validation loss {val_loss}")

    # Save the model at the last epoch regardless of validation loss
    if epoch == epoch - 1:  # num_epochs is the total number of epochs
        torch.save(checkpoint, os.path.join(save_path, 'final_model.pth'))
        print(f"Final model saved at epoch {epoch+1}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print('Early stopping triggered.')
            break


#Matrices and Evaluate

In [ ]:
import zipfile
import os


with zipfile.ZipFile('/content/zipped_windows.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/data')


In [ ]:
import torch
from torch.utils.data import Dataset
import json
import os

class JsonDataset(Dataset):
    def __init__(self, file_path):
        self.data_path = file_path

    def __len__(self):
        return len(os.listdir(self.data_path))

    def __getitem__(self, idx):
        file_path = self.data_path+f"windows_{idx}.json"
        with open(file_path, 'r') as f:
          data = json.load(f)
          input_ids = torch.tensor(data['input_ids'])
          attention_mask = torch.tensor(data['attention_mask'])
          labels = torch.tensor(data['labels'])
        return input_ids, attention_mask, labels



In [ ]:
import numpy as np

def compute_pk(true_labels, pred_labels, step=4):
    """
    Compute the PK metric.
    Args:
        true_labels (list): Ground truth binary segmentation labels.
        pred_labels (list): Predicted binary segmentation labels.
        step (int): Size of the window used for evaluation.
    Returns:
        float: PK score.
    """
    n = len(true_labels)
    mismatches = 0
    total_windows = 0

    for i in range(n - step):
        # Check for segmentation mismatch in the window
        true_boundary = true_labels[i] != true_labels[i + step]
        pred_boundary = pred_labels[i] != pred_labels[i + step]

        if true_boundary != pred_boundary:
            mismatches += 1
        total_windows += 1

    return mismatches / total_windows if total_windows > 0 else 1.0


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = BigModel().to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['robert

Initializing weights from pre-trained RoBERTa model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded RoBERTa model.
Embeddings initialized.
Layer 0: Attention weights initialized with shape:
  self.query.weight: torch.Size([768, 768])
  self.query.bias: torch.Size([768])
  self.key.weight: torch.Size([768, 768])
  self.key.bias: torch.Size([768])
  self.value.weight: torch.Size([768, 768])
  self.value.bias: torch.Size([768])
  output.dense.weight: torch.Size([768, 768])
  output.dense.bias: torch.Size([768])
  output.LayerNorm.weight: torch.Size([768])
  output.LayerNorm.bias: torch.Size([768])
Layer 0: FFN_1 weights initialized with shape:
  dense.weight: torch.Size([3072, 768])
  dense.bias: torch.Size([3072])
Layer 0: FFN_2 weights initialized with shape:
  dense.weight: torch.Size([768, 3072])
  dense.bias: torch.Size([768])
  LayerNorm.weight: torch.Size([768])
  LayerNorm.bias: torch.Size([768])
Layer 1: Attention weights initialized with shape:
  self.query.weight: torch.Size([768, 768])
  self.query.bias: torch.Size([768])
  self.key.weight: torch.Size([768, 768])
  se

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:

checkpoint_path = "/content/drive/MyDrive/best_model.pth"
model.load_state_dict(torch.load(checkpoint_path))



<ipython-input-30-facb0207fe4d>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path))


<All keys matched successfully>

In [ ]:
from torch.utils.data import DataLoader

# Define paths and parameters
dataset = JsonDataset(file_path="/content/data/")
data_loader = DataLoader(dataset, batch_size=1, shuffle=False)
window_size = 16  # Adjust this based on your task

# Placeholder for evaluation
all_pk_scores = []

# Evaluate
with torch.no_grad():
    for input_ids, attention_mask, true_labels in data_loader:
        # Pass input through the model
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        true_labels = true_labels.to(device)

        outputs = model(input_ids, attention_mask)

        # Assuming the model output needs a threshold to get binary predictions
        pred_labels = (outputs > 0.3).int()
        # print(pred_labels)
        # print(true_labels)


        # Convert to flat lists if needed
        true_labels = true_labels.flatten().tolist()
        pred_labels = pred_labels.flatten().tolist()

        # Compute PK for this example
        pk_score = compute_pk(true_labels, pred_labels)
        all_pk_scores.append(pk_score)

# Average PK score across the dataset
average_pk = np.mean(all_pk_scores)
print(f"Average PK score: {average_pk:.4f}")


Average PK score: 0.2062
